# Unified Formalization Analysis

Comprehensive visualization of the **Semantic Boundary Testing Framework** for Vision-Language Models.

This notebook implements the visualizations defined in the [Unified Formalization](../../docs/Unified_Formalization.md) and provides evidence for hypotheses **H1-H4**.

---

## Structure

The analysis follows the **3-level boundary framework**:

| Level | Space | Symbol | Section |
|-------|-------|--------|---------|
| 1 | Input (Semantic) | $\kappa: \mathcal{M} \to \mathcal{C}$ | §3 |
| 2 | Embedding | $d_B(z)$, Voronoi | §4 |
| 3 | Output (Behavioral) | $C: \mathcal{T}_{raw} \to \mathcal{O}$ | §7 |

## Hypotheses

| ID | Hypothesis | Status |
|----|------------|--------|
| H1 | Boundary-Error Correlation: $\text{Cor}(d_B, \epsilon_{\mathcal{O}}) < 0$ | Tested |
| H2 | Anisotropy: $\exists k_i, k_j : \bar{S}_{k_i} \gg \bar{S}_{k_j}$ | Tested |
| H3 | Asymmetry: $\exists (c_i, c_j) : S_{i \to j} \neq S_{j \to i}$ | Tested |
| H4 | Tier-Dependent Resolution | Pending |

---

In [1]:
# Core imports
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import ipywidgets as widgets
from IPython.display import display

# Visualization utilities
from archive.pipeline.notebooks.utils import (
    # Data
    load_pipeline_data,
    CLASSIFICATION_KEYS,
    THEME,
    format_pct,
    format_count,
    # Embedding
    create_3d_explorer,
    # Boundary
    create_boundary_geography,
    create_sharpness_histograms,
    create_cost_surface_grid,
    # Transition
    create_transition_sankey,
    create_ade_transition_matrix,
    identify_danger_zones,
    create_danger_zone_plot,
    # Pairs
    create_pair_scatter,
    create_pair_connections,
    # Hypothesis (H1-H4)
    compute_boundary_margin,
    create_h1_correlation_plot,
    create_h1_perkey_correlation_plot,
    compute_margin_sensitivity_alignment,
    compute_anisotropy_vector,
    create_h2_anisotropy_plot,
    create_h3_asymmetry_heatmap,
    create_h3_asymmetry_distribution,
    create_stability_map,
    create_divergence_curve_plot,
    create_three_level_summary,
    export_figure_for_print,
)

# Load data
data = load_pipeline_data()

print(f"Dataset Summary")
print(f"═" * 40)
print(f"Scenes:     {format_count(data.n_scenes):>8s}  ({format_count(data.n_with_ade)} with ADE)")
print(f"Pairs:      {format_count(data.n_pairs):>8s}  ({format_count(data.n_pairs_with_ade)} with ΔADE)")
print(f"Embeddings: {data.embeddings.shape[1]:>8d}-dim")
print(f"Keys:       {len(CLASSIFICATION_KEYS):>8d}")

Dataset Summary
════════════════════════════════════════
Scenes:        2,647  (947 with ADE)
Pairs:        13,043  (4,270 with ΔADE)
Embeddings:     1280-dim
Keys:              6


---

## Global Filters

Configure analysis scope. Changes apply to all sections below.

In [2]:
# Global filter state
class Filters:
    def __init__(self):
        self.focus_key = widgets.Dropdown(
            options=["All"] + CLASSIFICATION_KEYS,
            value="All",
            description="Focus Key:",
            style={"description_width": "80px"},
        )
        self.min_delta = widgets.FloatSlider(
            value=0, min=0, max=2, step=0.1,
            description="Min ΔADE:",
            style={"description_width": "80px"},
        )
        self.export_mode = widgets.Checkbox(
            value=False,
            description="Export Mode (static figures)",
        )
    
    def get_filtered_pairs(self):
        df = data.pairs.copy()
        if self.focus_key.value != "All":
            df = df[df["diff_key"] == self.focus_key.value]
        if self.min_delta.value > 0:
            df = df[df["rel_delta_ade"] >= self.min_delta.value]
        return df

filters = Filters()

display(widgets.HBox([
    filters.focus_key,
    filters.min_delta,
    filters.export_mode,
]))

---

# Level 1: Input Space (Semantic Structure)

The semantic key system $\kappa: \mathcal{M} \to \mathcal{C}$ partitions inputs by human-defined semantics.

$$\mathcal{C} = \prod_{k \in \mathcal{K}} V_k$$

---

## 1.1 Semantic Key Distribution

Distribution of semantic attribute values across the dataset.

In [3]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from archive.pipeline.notebooks.utils import plotly_layout, axis_style

n_keys = len(CLASSIFICATION_KEYS)
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=CLASSIFICATION_KEYS,
    vertical_spacing=0.15,
    horizontal_spacing=0.1,
)

for i, key in enumerate(CLASSIFICATION_KEYS):
    row = i // 3 + 1
    col = i % 3 + 1
    
    counts = data.scenes[key].value_counts().sort_index()
    
    fig.add_trace(
        go.Bar(
            x=counts.index.tolist(),
            y=counts.values,
            marker_color=THEME["categorical"][i],
            showlegend=False,
        ),
        row=row, col=col,
    )

fig.update_layout(
    **plotly_layout("Semantic Key Distributions (Level 1: κ)", height=500, show_legend=False),
)
fig.update_xaxes(tickangle=45, tickfont_size=9)
fig.show()

if filters.export_mode.value:
    export_figure_for_print(fig, "fig1_key_distributions")

## 1.2 Class Space Cardinality

$$|\mathcal{C}| = \prod_{i=1}^{m} |V_{k_i}|$$

In [4]:
cardinality = 1
print("Key Cardinalities |V_k|:")
print("═" * 40)
for key in CLASSIFICATION_KEYS:
    n_values = data.scenes[key].nunique()
    cardinality *= n_values
    print(f"  {key:20s} {n_values:4d} values")

print("═" * 40)
print(f"  Total |C|:          {cardinality:,d} possible classes")
print(f"  Observed classes:   {len(data.scenes.groupby(CLASSIFICATION_KEYS)):,d}")

Key Cardinalities |V_k|:
════════════════════════════════════════
  weather                 4 values
  time_of_day             3 values
  depth_complexity        2 values
  occlusion_level         4 values
  road_type               5 values
  required_action         4 values
════════════════════════════════════════
  Total |C|:          1,920 possible classes
  Observed classes:   367


---

# Level 2: Embedding Space (Representation Geometry)

The embedding space $\mathcal{Z} = E(\mathcal{X}) \subset \mathbb{R}^{d_e}$ provides continuous geometry.

Key metrics:
- **Boundary proximity** $d_B(z)$: Distance to nearest Voronoi boundary
- **Sharpness**: Proportion of k-NN with different labels

---

## 2.1 3D Embedding Explorer

Interactive exploration of the embedding space. Point size encodes ADE; color encodes selected attribute.

In [5]:
color_options = ["ade_class", "is_anchor", "label_confidence"] + CLASSIFICATION_KEYS

dropdown = widgets.Dropdown(
    options=color_options,
    value="ade_class",
    description="Color by:",
)
output = widgets.Output()

def update_explorer(change):
    with output:
        output.clear_output(wait=True)
        fig = create_3d_explorer(data.scenes, color_by=change["new"])
        display(fig)
        if filters.export_mode.value:
            export_figure_for_print(fig, f"fig2_embedding_{change['new']}")

dropdown.observe(update_explorer, names="value")
display(dropdown)

with output:
    display(create_3d_explorer(data.scenes, color_by="ade_class"))
display(output)

Dropdown(description='Color by:', options=('ade_class', 'is_anchor', 'label_confidence', 'weather', 'time_of_d…

Output()

## 2.2 Boundary Geography

Where do semantic boundaries concentrate in embedding space?

In [6]:
dropdown = widgets.Dropdown(
    options=["All"] + CLASSIFICATION_KEYS,
    value="All",
    description="Key:",
)
output = widgets.Output()

def update_geography(change):
    with output:
        output.clear_output(wait=True)
        key = None if change["new"] == "All" else change["new"]
        fig = create_boundary_geography(data.scenes, data.pairs, key=key)
        display(fig)

dropdown.observe(update_geography, names="value")
display(dropdown)

with output:
    display(create_boundary_geography(data.scenes, data.pairs))
display(output)

Dropdown(description='Key:', options=('All', 'weather', 'time_of_day', 'depth_complexity', 'occlusion_level', …

Output()

## 2.3 Boundary Sharpness (k-NN)

Sharpness = proportion of k-NN neighbors with different label.
- **0.0** = perfectly crisp boundary
- **1.0** = maximally fuzzy

In [7]:
fig = create_sharpness_histograms(data.scenes, data.embeddings, k=10)
fig.show()

if filters.export_mode.value:
    export_figure_for_print(fig, "fig3_sharpness_histograms")

## 2.4 Cost Surfaces

Background: interpolated ADE. Contours: decision boundary entropy.

In [8]:
fig = create_cost_surface_grid(data.scenes)
fig.show()

if filters.export_mode.value:
    export_figure_for_print(fig, "fig4_cost_surfaces")

---

# Level 3: Output Space (Behavioral Boundaries)

The trajectory classifier $C: \mathcal{T}_{raw} \to \mathcal{O}$ detects behavioral changes.

$$\mathcal{O} = C_{dir} \times C_{speed} \times C_{lat}$$

---

## 3.1 Stability Map $\mathcal{S}$

Primary summary: sensitivity ranking by semantic key.

In [9]:
fig = create_stability_map(data.pairs, show_change_rate=True)
fig.show()

if filters.export_mode.value:
    export_figure_for_print(fig, "fig5_stability_map")

## 3.2 Transition Flows (Sankey)

Edge width = pair count. Edge color = mean ΔADE severity.

In [10]:
dropdown = widgets.Dropdown(
    options=CLASSIFICATION_KEYS,
    value="depth_complexity",
    description="Key:",
)
output = widgets.Output()

def update_sankey(change):
    with output:
        output.clear_output(wait=True)
        fig = create_transition_sankey(data.pairs, change["new"])
        display(fig)

dropdown.observe(update_sankey, names="value")
display(dropdown)

with output:
    display(create_transition_sankey(data.pairs, "depth_complexity"))
display(output)

Dropdown(description='Key:', index=2, options=('weather', 'time_of_day', 'depth_complexity', 'occlusion_level'…

Output()

## 3.3 ADE Class Transition Matrix

How often do semantic boundary crossings cause ADE class changes?

In [11]:
fig = create_ade_transition_matrix(data.pairs)
fig.show()

# Per-key breakdown
print("\nADE Class Change Rate by Key:")
print("═" * 45)
for key in CLASSIFICATION_KEYS:
    df_key = data.pairs[(data.pairs["diff_key"] == key) & (data.pairs["rel_delta_ade"].notna())]
    if len(df_key) > 0:
        rate = df_key["ade_class_changed"].mean()
        print(f"  {key:22s} {format_pct(rate):>6s}  (n={len(df_key)})")

if filters.export_mode.value:
    export_figure_for_print(fig, "fig6_ade_transitions")


ADE Class Change Rate by Key:
═════════════════════════════════════════════
  weather                 63.8%  (n=437)
  time_of_day             70.7%  (n=140)
  depth_complexity        64.4%  (n=329)
  occlusion_level         69.7%  (n=429)
  road_type               70.1%  (n=572)
  required_action         65.9%  (n=745)


## 3.4 Danger Zones

Scenes near boundaries AND with high ADE class change rate.

In [12]:
danger_df = identify_danger_zones(
    data.scenes, data.embeddings, data.pairs,
    k=10, boundary_threshold=0.3, ade_change_threshold=0.5
)

n_danger = danger_df["is_danger"].sum()
affected_pairs = data.pairs[
    (data.pairs["clip_a"].isin(danger_df[danger_df["is_danger"]]["clip_id"])) |
    (data.pairs["clip_b"].isin(danger_df[danger_df["is_danger"]]["clip_id"]))
]

print(f"Danger Zone Summary")
print(f"═" * 40)
print(f"  Danger zones:    {n_danger:>6d} scenes")
print(f"  Affected pairs:  {len(affected_pairs):>6d} ({format_pct(len(affected_pairs)/len(data.pairs))})")

fig = create_danger_zone_plot(danger_df)
fig.show()

if filters.export_mode.value:
    export_figure_for_print(fig, "fig7_danger_zones")

Danger Zone Summary
════════════════════════════════════════
  Danger zones:       402 scenes
  Affected pairs:    3080 (23.6%)


---

# Hypothesis Testing

Evidence for the central hypotheses from §12 of the Unified Formalization.

---

## H1: Boundary-Error Correlation

$$\text{Cor}(d_B(z), \epsilon_{\mathcal{O}}(z)) < 0$$

Scenes whose embeddings lie closer to Voronoi boundaries have higher trajectory prediction errors.

In [13]:
fig = create_h1_correlation_plot(
    data.scenes,
    data.embeddings,
    k=10,
    show_regression=True,
    show_confidence=True,
)
fig.show()

if filters.export_mode.value:
    export_figure_for_print(fig, "fig8_h1_correlation", width=700, height=500)

### H1: Per-Key Decomposition

The boundary margin is naturally a **vector**, not a scalar:

$$\mathbf{d}_B(z) = \big(d_B^{\text{weather}}(z),\ d_B^{\text{time}}(z),\ d_B^{\text{depth}}(z),\ d_B^{\text{occ}}(z),\ d_B^{\text{road}}(z),\ d_B^{\text{action}}(z)\big)$$

Testing $\text{Cor}(d_B^k, \text{ADE})$ for each key $k$ reveals whether H1 fails uniformly or varies by boundary type.

In [14]:
fig = create_h1_perkey_correlation_plot(data.scenes, data.embeddings)
fig.show()

if filters.export_mode.value:
    export_figure_for_print(fig, "fig09_h1_perkey_correlation", width=700, height=400)

### H1→H2 Bridge: Margin-Sensitivity Alignment

Do keys with tighter boundaries (low $\bar{d}_B^k$) have higher sensitivity (high $S_k$)?

This connects the embedding-space geometry (H1) with the behavioral anisotropy (H2).

In [15]:
import plotly.graph_objects as go

# Compute alignment between margin and sensitivity vectors
alignment_df = compute_margin_sensitivity_alignment(data.scenes, data.pairs, data.embeddings)

# Display table
print("Margin-Sensitivity Alignment:")
print("═" * 55)
print(f"{'Key':20s} {'Mean d_B^k':>12s} {'Sensitivity':>12s}")
print("─" * 55)
for _, row in alignment_df.sort_values("mean_margin").iterrows():
    print(f"  {row['key']:18s} {row['mean_margin']:>10.4f} {row['sensitivity']:>12.3f}")
print("─" * 55)
rho = alignment_df.attrs.get("spearman_rho", float("nan"))
p = alignment_df.attrs.get("spearman_p", float("nan"))
print(f"  Spearman ρ = {rho:.3f} (p = {p:.3f})")
print()
print("  Interpretation: ρ < 0 means tighter boundaries → higher sensitivity")

# Scatter plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=alignment_df["mean_margin"],
    y=alignment_df["sensitivity"],
    mode="markers+text",
    marker=dict(size=12, color=THEME["text"]),
    text=alignment_df["key"],
    textposition="top center",
    textfont=dict(size=9),
    hovertemplate="<b>%{text}</b><br>d_B: %{x:.4f}<br>S: %{y:.3f}<extra></extra>",
))

# Add regression line
x = alignment_df["mean_margin"].values
y = alignment_df["sensitivity"].values
slope, intercept = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 50)
fig.add_trace(go.Scatter(
    x=x_line,
    y=slope * x_line + intercept,
    mode="lines",
    line=dict(color=THEME["text_secondary"], width=2, dash="dash"),
    showlegend=False,
))

# Annotation
fig.add_annotation(
    x=0.98, y=0.98,
    xref="paper", yref="paper",
    text=f"ρ = {rho:.2f}",
    showarrow=False,
    font=dict(size=12, family=THEME["font_mono"]),
    xanchor="right", yanchor="top",
)

fig.update_layout(
    **plotly_layout("", height=400, show_legend=False, margin=dict(l=60, r=30, t=30, b=60)),
    xaxis=axis_style("Mean Boundary Margin d̄_B^k"),
    yaxis=axis_style("Sensitivity S_k"),
)
fig.show()

if filters.export_mode.value:
    export_figure_for_print(fig, "fig10_margin_sensitivity_alignment", width=600, height=400)

Margin-Sensitivity Alignment:
═══════════════════════════════════════════════════════
Key                    Mean d_B^k  Sensitivity
───────────────────────────────────────────────────────
  required_action        0.0092        0.923
  occlusion_level        0.0108        0.924
  depth_complexity       0.0131        0.990
  weather                0.0159        0.869
  road_type              0.0239        0.884
  time_of_day            0.0410        0.703
───────────────────────────────────────────────────────
  Spearman ρ = -0.714 (p = 0.111)

  Interpretation: ρ < 0 means tighter boundaries → higher sensitivity


## H2: Anisotropy

$$\exists k_i, k_j \in \mathcal{K} : \bar{S}_{k_i} \gg \bar{S}_{k_j}$$

The VLM's decision landscape is anisotropic — sensitivity varies by semantic axis.

In [16]:
# Compute anisotropy metrics
means, stds, alpha = compute_anisotropy_vector(data.pairs)

print(f"H2: Anisotropy Analysis")
print(f"═" * 40)
print(f"  Anisotropy Ratio α = {alpha:.2f}")
print(f"  (α = 1.0 implies isotropy; α >> 1 implies anisotropy)")
print()
print(f"  Mean Sensitivity by Key:")
for key in sorted(means.keys(), key=lambda k: means[k], reverse=True):
    print(f"    {key:20s} {means[key]:.3f} ± {stds[key]:.3f}")

# Bar chart visualization
fig = create_h2_anisotropy_plot(data.pairs, style="bar")
fig.show()

if filters.export_mode.value:
    export_figure_for_print(fig, "fig9_h2_anisotropy", width=700, height=400)

H2: Anisotropy Analysis
════════════════════════════════════════
  Anisotropy Ratio α = 1.41
  (α = 1.0 implies isotropy; α >> 1 implies anisotropy)

  Mean Sensitivity by Key:
    depth_complexity     0.990 ± 0.613
    occlusion_level      0.924 ± 0.554
    required_action      0.923 ± 0.560
    road_type            0.884 ± 0.548
    weather              0.869 ± 0.540
    time_of_day          0.703 ± 0.541


### H2: Radar View

Alternative polar visualization of the anisotropy vector $\mathbf{a}$.

In [17]:
fig = create_h2_anisotropy_plot(data.pairs, style="radar")
fig.show()

## H3: Asymmetry

$$\exists (c_i, c_j) : S_{i \to j} \neq S_{j \to i}$$

Transitions are not symmetric — the model treats direction differently.

In [18]:
dropdown = widgets.Dropdown(
    options=CLASSIFICATION_KEYS,
    value="weather",
    description="Key:",
)
output = widgets.Output()

def update_asymmetry(change):
    with output:
        output.clear_output(wait=True)
        fig = create_h3_asymmetry_heatmap(data.pairs, change["new"])
        display(fig)
        if filters.export_mode.value:
            export_figure_for_print(fig, f"fig10_h3_asymmetry_{change['new']}")

dropdown.observe(update_asymmetry, names="value")
display(dropdown)

with output:
    display(create_h3_asymmetry_heatmap(data.pairs, "weather"))
display(output)

Dropdown(description='Key:', options=('weather', 'time_of_day', 'depth_complexity', 'occlusion_level', 'road_t…

Output()

### H3: Asymmetry Distribution

Distribution of $S_{ij} - S_{ji}$ across all transitions. If symmetric, this would be centered at 0.

In [19]:
fig = create_h3_asymmetry_distribution(data.pairs)
fig.show()

if filters.export_mode.value:
    export_figure_for_print(fig, "fig11_h3_asymmetry_dist")

---

# Divergence Curves (§8)

$$\delta(t) = D(y(t), y(0))$$

How output divergence changes along interpolation paths between class centroids.

In [20]:
dropdown = widgets.Dropdown(
    options=CLASSIFICATION_KEYS,
    value="depth_complexity",
    description="Key:",
)
output = widgets.Output()

def update_divergence(change):
    with output:
        output.clear_output(wait=True)
        fig = create_divergence_curve_plot(data.pairs, change["new"], n_curves=5)
        display(fig)

dropdown.observe(update_divergence, names="value")
display(dropdown)

with output:
    display(create_divergence_curve_plot(data.pairs, "depth_complexity", n_curves=5))
display(output)

Dropdown(description='Key:', index=2, options=('weather', 'time_of_day', 'depth_complexity', 'occlusion_level'…

Output()

---

# Pair Deep Dive

Detailed examination of individual boundary crossings.

---

## Pair ADE Comparison

Points away from the diagonal indicate high sensitivity to semantic change.

In [21]:
key_dropdown = widgets.Dropdown(
    options=["All"] + CLASSIFICATION_KEYS,
    value="All",
    description="Key:",
)
delta_slider = widgets.FloatSlider(
    value=0, min=0, max=2, step=0.1,
    description="Min ΔADE:",
)
output = widgets.Output()

def update_pair_scatter(change=None):
    with output:
        output.clear_output(wait=True)
        key = key_dropdown.value if key_dropdown.value != "All" else None
        fig = create_pair_scatter(data.pairs, key=key, min_delta=delta_slider.value)
        display(fig)

key_dropdown.observe(update_pair_scatter, names="value")
delta_slider.observe(update_pair_scatter, names="value")

display(widgets.HBox([key_dropdown, delta_slider]))
update_pair_scatter()
display(output)

Output()

## Pair Connections (3D)

Spatial structure of high-ΔADE boundary crossings in embedding space.

In [22]:
dropdown = widgets.Dropdown(
    options=CLASSIFICATION_KEYS,
    value="depth_complexity",
    description="Key:",
)
output = widgets.Output()

def update_connections(change):
    with output:
        output.clear_output(wait=True)
        fig = create_pair_connections(
            data.scenes, data.pairs, change["new"],
            max_pairs=50, use_3d=True
        )
        display(fig)

dropdown.observe(update_connections, names="value")
display(dropdown)

with output:
    display(create_pair_connections(data.scenes, data.pairs, "depth_complexity", max_pairs=50, use_3d=True))
display(output)

Dropdown(description='Key:', index=2, options=('weather', 'time_of_day', 'depth_complexity', 'occlusion_level'…

Output()

---

# Summary: Three-Level Alignment

Overview of the three boundary levels from §11.

---

In [23]:
fig = create_three_level_summary(data.scenes, data.pairs, data.embeddings)
fig.show()

if filters.export_mode.value:
    export_figure_for_print(fig, "fig12_three_level_summary", width=900, height=350)

## Key Findings Summary

In [24]:
from scipy import stats

# Compute summary statistics
means, stds, alpha = compute_anisotropy_vector(data.pairs)

# H1: Correlation (using centroid-based margin per §4.3)
mask = data.scenes["ade"].notna()
if mask.sum() > 10:
    scenes_with_ade = data.scenes[mask].copy().reset_index(drop=True)
    margin_result = compute_boundary_margin(scenes_with_ade, data.embeddings, return_full=True)
    d_B = margin_result.mean
    valid = ~np.isnan(d_B)
    if valid.sum() > 10:
        r, p = stats.pearsonr(d_B[valid], scenes_with_ade.loc[valid, "ade"].values)
    else:
        r, p = 0, 1
    
    # Per-key correlations
    perkey_cors = {}
    ade_vals = scenes_with_ade["ade"].values
    for key in CLASSIFICATION_KEYS:
        d_B_k = margin_result.per_key[key]
        valid_k = ~np.isnan(d_B_k)
        if valid_k.sum() >= 10:
            r_k, p_k = stats.pearsonr(d_B_k[valid_k], ade_vals[valid_k])
            perkey_cors[key] = {"r": r_k, "p": p_k}
else:
    r, p = 0, 1
    perkey_cors = {}

# Margin-sensitivity alignment
alignment = compute_margin_sensitivity_alignment(data.scenes, data.pairs, data.embeddings)
rho_align = alignment.attrs.get("spearman_rho", float("nan"))
p_align = alignment.attrs.get("spearman_p", float("nan"))

# Overall ADE change rate
df_ade = data.pairs[data.pairs["rel_delta_ade"].notna()]
overall_change_rate = df_ade["ade_class_changed"].mean()

# Most sensitive key
top_key = max(means.keys(), key=lambda k: means[k])
bottom_key = min(means.keys(), key=lambda k: means[k])

print("═" * 65)
print("                      KEY FINDINGS SUMMARY")
print("═" * 65)
print()
print(f"  H1: Boundary-Error Correlation (Aggregate)")
print(f"      Pearson r = {r:.3f} (p = {p:.4f})")
print(f"      {'✓ SUPPORTED' if r < 0 and p < 0.05 else '✗ NOT SUPPORTED'}")
print()
print(f"  H1: Per-Key Decomposition")
for key in CLASSIFICATION_KEYS:
    if key in perkey_cors:
        r_k, p_k = perkey_cors[key]["r"], perkey_cors[key]["p"]
        sig = "**" if p_k < 0.01 else "*" if p_k < 0.05 else ""
        print(f"      {key:20s} r = {r_k:+.3f}{sig}")
print()
print(f"  H1→H2 Bridge: Margin-Sensitivity Alignment")
print(f"      Spearman ρ = {rho_align:.3f} (p = {p_align:.3f})")
print(f"      {'✓ Tighter boundaries → higher sensitivity' if rho_align < -0.5 else '~ Weak/no alignment'}")
print()
print(f"  H2: Anisotropy")
print(f"      Anisotropy ratio α = {alpha:.2f}")
print(f"      Most sensitive:  {top_key} ({means[top_key]:.3f})")
print(f"      Least sensitive: {bottom_key} ({means[bottom_key]:.3f})")
print(f"      {'✓ SUPPORTED' if alpha > 1.2 else '✗ NOT SUPPORTED'} (moderate anisotropy)")
print()
print(f"  H3: Asymmetry")
print(f"      (See heatmaps above for evidence)")
print()
print(f"  Overall Statistics:")
print(f"      ADE class change rate: {format_pct(overall_change_rate)}")
print(f"      Danger zones: {danger_df['is_danger'].sum()} scenes")
print()
print("═" * 65)

═════════════════════════════════════════════════════════════════
                      KEY FINDINGS SUMMARY
═════════════════════════════════════════════════════════════════

  H1: Boundary-Error Correlation (Aggregate)
      Pearson r = -0.006 (p = 0.8638)
      ✗ NOT SUPPORTED

  H1: Per-Key Decomposition
      weather              r = +0.045
      time_of_day          r = -0.016
      depth_complexity     r = +0.014
      occlusion_level      r = -0.008
      road_type            r = -0.021
      required_action      r = -0.007

  H1→H2 Bridge: Margin-Sensitivity Alignment
      Spearman ρ = -0.714 (p = 0.111)
      ✓ Tighter boundaries → higher sensitivity

  H2: Anisotropy
      Anisotropy ratio α = 1.41
      Most sensitive:  depth_complexity (0.990)
      Least sensitive: time_of_day (0.703)
      ✓ SUPPORTED (moderate anisotropy)

  H3: Asymmetry
      (See heatmaps above for evidence)

  Overall Statistics:
      ADE class change rate: 66.7%
      Danger zones: 402 scenes

══

---

# Export All Figures

Generate publication-quality static figures for thesis.

In [25]:
def export_all_figures():
    """Export all key figures for thesis."""
    import os
    os.makedirs("figures", exist_ok=True)
    
    print("Exporting figures for thesis...")
    print("═" * 50)
    
    # Figure 1: Three-level summary
    fig = create_three_level_summary(data.scenes, data.pairs, data.embeddings)
    export_figure_for_print(fig, "fig01_three_level_summary", width=900, height=350)
    
    # Figure 2: Stability map
    fig = create_stability_map(data.pairs, show_change_rate=True)
    export_figure_for_print(fig, "fig02_stability_map", width=900, height=400)
    
    # Figure 3: H1 correlation (aggregate)
    fig = create_h1_correlation_plot(data.scenes, data.embeddings, k=10)
    export_figure_for_print(fig, "fig03_h1_correlation", width=700, height=500)
    
    # Figure 4: H1 per-key decomposition
    fig = create_h1_perkey_correlation_plot(data.scenes, data.embeddings)
    export_figure_for_print(fig, "fig04_h1_perkey_correlation", width=700, height=400)
    
    # Figure 5: Margin-sensitivity alignment
    alignment_df = compute_margin_sensitivity_alignment(data.scenes, data.pairs, data.embeddings)
    rho = alignment_df.attrs.get("spearman_rho", float("nan"))
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=alignment_df["mean_margin"],
        y=alignment_df["sensitivity"],
        mode="markers+text",
        marker=dict(size=12, color=THEME["text"]),
        text=alignment_df["key"],
        textposition="top center",
        textfont=dict(size=9),
    ))
    x = alignment_df["mean_margin"].values
    y = alignment_df["sensitivity"].values
    slope, intercept = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 50)
    fig.add_trace(go.Scatter(
        x=x_line, y=slope * x_line + intercept,
        mode="lines",
        line=dict(color=THEME["text_secondary"], width=2, dash="dash"),
        showlegend=False,
    ))
    fig.add_annotation(x=0.98, y=0.98, xref="paper", yref="paper",
        text=f"ρ = {rho:.2f}", showarrow=False,
        font=dict(size=12, family=THEME["font_mono"]), xanchor="right", yanchor="top")
    fig.update_layout(
        **plotly_layout("", height=400, show_legend=False, margin=dict(l=60, r=30, t=30, b=60)),
        xaxis=axis_style("Mean Boundary Margin d̄_B^k"),
        yaxis=axis_style("Sensitivity S_k"),
    )
    export_figure_for_print(fig, "fig05_margin_sensitivity_alignment", width=600, height=400)
    
    # Figure 6: H2 anisotropy
    fig = create_h2_anisotropy_plot(data.pairs, style="bar")
    export_figure_for_print(fig, "fig06_h2_anisotropy", width=700, height=400)
    
    # Figure 7: H3 asymmetry (weather as example)
    fig = create_h3_asymmetry_heatmap(data.pairs, "weather")
    export_figure_for_print(fig, "fig07_h3_asymmetry_weather", width=600, height=450)
    
    # Figure 8: Danger zones
    fig = create_danger_zone_plot(danger_df)
    export_figure_for_print(fig, "fig08_danger_zones", width=700, height=500)
    
    # Figure 9: ADE transitions
    fig = create_ade_transition_matrix(data.pairs)
    export_figure_for_print(fig, "fig09_ade_transitions", width=500, height=400)
    
    # Figure 10: Sharpness histograms
    fig = create_sharpness_histograms(data.scenes, data.embeddings, k=10)
    export_figure_for_print(fig, "fig10_sharpness_histograms", width=900, height=500)
    
    print("\n✓ All figures exported to ./figures/")

# Uncomment to export:
export_all_figures()

Exporting figures for thesis...
══════════════════════════════════════════════════
Exported: figures/fig01_three_level_summary.{png,pdf,html}
Exported: figures/fig02_stability_map.{png,pdf,html}
Exported: figures/fig03_h1_correlation.{png,pdf,html}
Exported: figures/fig04_h1_perkey_correlation.{png,pdf,html}
Exported: figures/fig05_margin_sensitivity_alignment.{png,pdf,html}
Exported: figures/fig06_h2_anisotropy.{png,pdf,html}
Exported: figures/fig07_h3_asymmetry_weather.{png,pdf,html}
Exported: figures/fig08_danger_zones.{png,pdf,html}
Exported: figures/fig09_ade_transitions.{png,pdf,html}
Exported: figures/fig10_sharpness_histograms.{png,pdf,html}

✓ All figures exported to ./figures/


---

*Generated by the Semantic Boundary Testing Framework*

Reference: [Unified Formalization](../../docs/Unified_Formalization.md) §1-17